# Feature Selection Analysis on the Pokemon Dataset
I will analyze the Pokemon dataset, focusing on comparing different feature selection methods from the kydavra library. My goal is to evaluate how these selectors impact the accuracy and precision of a DecisionTreeClassifier in predicting legendary status.

# Imports and Setup
I am using pandas for data handling, sklearn for modeling, and kydavra selectors. LabelEncoder is also important for converting categorical data

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score
from sklearn.preprocessing import LabelEncoder
from kydavra import PValueSelector, LassoSelector, PearsonCorrelationSelector, PointBiserialCorrSelector

import warnings

warnings.filterwarnings("ignore")

# Loading the Dataset and preparing my Data
I load the Pokemon.csv file, drop the # and Name columns, and fill missing Type 2 values with "None". I then use LabelEncoder to convert all object and boolean columns into numeric values so the selectors can process them. My target variable for this run is Type 1.

In [10]:
df = pd.read_csv("../Datasets/Outliers1/Pokemon.csv")
df.head()

,#,Name,Type 1,Type 2,Total,HP,Attack,Defense,Sp. Atk,Sp. Def,Speed,Generation,Legendary
0,1,Bulbasaur,Grass,Poison,318,45,49,49,65,65,45,1,False
1,2,Ivysaur,Grass,Poison,405,60,62,63,80,80,60,1,False
2,3,Venusaur,Grass,Poison,525,80,82,83,100,100,80,1,False
3,3,VenusaurMega Venusaur,Grass,Poison,625,80,100,123,122,120,80,1,False
4,4,Charmander,Fire,NaN,309,39,52,43,60,50,65,1,False


In [11]:
df = df.drop(['#', 'Name'], axis=1).fillna("None")

In [12]:
le = LabelEncoder()
for col in df.select_dtypes(include=['object', 'bool']).columns:
    df[col] = le.fit_transform(df[col])

target = 'Type 1'

# Model Evaluation Function
I defined this helper function to handle the training and evaluation steps cleanly. It takes the DataFrame and target column, splits the data, trains a DecisionTreeClassifier, and returns the accuracy and weighted precision scores.

In [13]:
def evaluate_model(data, target_col):
    X = data.drop(target_col, axis=1)
    y = data[target_col]
    if X.empty: return 0.0, 0.0
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    return accuracy_score(y_test, y_pred), precision_score(y_test, y_pred, average='weighted', zero_division=0)

# Running the Feature Selection Analysis
I've listed the selectors I want to compare (PValueSelector, LassoSelector, PearsonCorrelationSelector, and PointBiserialCorrSelector). I iterate through them, call the .select() method, evaluate the model on the resulting features, and compile the scores. The output shows that PearsonCorrelationSelector did not select any features with its default settings, resulting in zero accuracy and precision

In [14]:
results = []
selectors = [
    ("PValueSelector", PValueSelector()),
    ("LassoSelector", LassoSelector()),
    ("PearsonCorrelationSelector", PearsonCorrelationSelector()), 
    ("PointBiserialCorrSelector", PointBiserialCorrSelector())
]

In [15]:
for name, selector in selectors:
    try:
        selected_cols = selector.select(df, target)
        
        if not selected_cols:
            print(f"Notice: {name} returned 0 features with default settings.")
            acc, prec = 0.0, 0.0
        else:
            acc, prec = evaluate_model(df[selected_cols + [target]], target)
            
        results.append({"Selector": name, "Accuracy": acc, "Precision": prec})
    except Exception as e:
        print(f"Error running {name}: {e}")
        results.append({"Selector": name, "Accuracy": 0.0, "Precision": 0.0})

Notice: PearsonCorrelationSelector returned 0 features with default settings.


In [16]:
results_df = pd.DataFrame(results)
print("\nFeature Selection Performance Summary:")
display(results_df.set_index("Selector").style.background_gradient(cmap='Blues').format("{:.4f}"))


Feature Selection Performance Summary:


,Accuracy,Precision
Selector,,
PValueSelector,0.1500,0.1632
LassoSelector,0.1688,0.1859
PearsonCorrelationSelector,0.0000,0.0000
PointBiserialCorrSelector,0.1562,0.1805


# Results Conclusion

Based on the Feature Selection Performance Summary, I have analyzed how different selectors impacted the model's ability to predict the target variable. The results show that LassoSelector achieved the highest performance across both metrics, with an Accuracy of 0.1688 and Precision of 0.1859. It was closely followed by the PointBiserialCorrSelector (0.1562 accuracy) and the PValueSelector (0.1500 accuracy).

> Key Takeaways

- Top Performer: The LassoSelector is currently the most effective method for this specific dataset and target, likely because it effectively zeroes out less impactful features through regularization.

- Selector Failure: The PearsonCorrelationSelector failed to select any features (0.0000), which happens when no features meet the default correlation threshold with the target.

- Overall Accuracy: The scores are generally low (under 17%), suggesting that the current features might not be strong predictors for the target or that the model needs further hyperparameter tuning.